# Layerwise ↔ FPM 对齐调查（移交 notebook）

- 模型:**Qwen3.6-35B-A3B**（数据是 3.6,不是 3.5）,b300,vLLM 0.20.1,tp4/ep4
- 分支:`simonec/layerwise-moe`。每节 = 问题 / 数据 / 结论。可复现（读本地 golden）
- **移交**:最后两节是「当前状态 + open items + 下一步」——新 session 从那看起
- 关键 doc(同目录):`SMALL_PREFILL_FIX_DESIGN.md`(主)· `MOE_INVESTIGATION_SUMMARY.md`(导航)
  · `runs/BUSY_METRIC_VERDICT.md` · `runs/SMALL_PREFILL_GAP_MICROCAUSALITY.md` · `DECODE_MOE_OVERLAY_VERDICT.md`

In [ ]:
import pandas as pd, numpy as np, subprocess
from pathlib import Path
REPO = Path.cwd()
while not (REPO/"src/aiconfigurator").exists() and REPO != REPO.parent: REPO = REPO.parent
GOLDEN   = REPO/"fpm_golden_runs/fpm_upfront_qwen36_moe_full_once_20260613_201336/tp4_ep4_past4096/fpm_metrics_phase.csv"
BACKBONE = REPO/"runs/layerwise_qwen36_tp4ep4_cleanctx4/layerwise_native_tagged128.csv"
g = pd.read_csv(GOLDEN)
print("golden 步数:", g.phase.value_counts().to_dict())

## 大框架

- 每个 step = **backbone(非-MoE)** + **MoE overlay**;step phase = decode / context / mixed
- 4 个方向,各打在不同层:

| # | 怀疑 | 哪层 | 结论 |
|---|---|---|---|
| 1 | expert 不均衡 / router | overlay | ❌ step-to-step skew 自平均,power-law 够 |
| 2 | layerwise≠FPM 口径 | 测量 | ❌ 只一小部分(ctx 计时,已修) |
| 3 | **MoE overlay 不准** | overlay | ✅ **已修+GPU验**(decode 51→21%,commit 4d825de3) |
| 4 | **small-prefill / mixed** | backbone | 🔬 **诊断到底**(下面),fix = serving-fidelity calibration,deferred |

## golden 是啥

- **受控 sweep,不是生产 trace**;6945 步 = 70 ctx + 300 mixed + 6575 decode
- 前几十步故意全跑 pure prefill(无 decode)来干净测 context
- prefill 切块:**大首块 + 小尾块**(528 对齐,block_size)
- ⚠️ golden 有 `--enable-expert-parallel` 但 **无 `--enable-eplb`** → EP routing 是 **skew** 的

In [ ]:
print(g[['phase','ctx_tokens','ctx_kv_tokens','decode_tokens']].head(6).to_string(index=False))
ctx = g[g.phase=='context']
print("\nctx_requests:", ctx.ctx_requests.value_counts().to_dict(), " (context 全 1 = 单请求)")
print("mixed ctx_requests 1~10 → mixed 的 ctx_tokens 是多请求和")

## #3 — MoE overlay decode 重复计(✅ 已修)

- **问题**:decode MoE 过预测 ~10×
- **原因**:① diagnostics `moe_inter` 写成 256(真 512)→ 命中 lower-level data ② fused kernel 已含 router/dispatch,AIC 又加 = double-count
- **修**:inter→512 + 去 `is_context` guard(kernel-source-aware,module_level 才删)
- **结果**:**decode MAPE 51→21%**,ratio 1.5→1.1-1.27;动吞吐 → 真赢点。已 commit + GPU 验
- **残余拆**(bs1):backbone +0.45ms(~13%)+ ep_a2a comm +0.39ms;routed 本身**准**(0.157≈golden 0.16)

In [ ]:
print("decode MAPE (qwen36 tp4_ep4):  BEFORE 50.9%   AFTER 19.3%")
print("mixed  MAPE:                   BEFORE 53.6%   AFTER 38.8%")
print("\n复现 AFTER(当前代码):")
cmd=["uv","run","python","collector/layerwise/diagnostics/compare_aic_layerwise_fpm_summary.py",
     "--layerwise",str(BACKBONE),"--moe-perf-file","collector/layerwise/wip/moe_perf.txt"]
out=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True).stdout
print("\n".join(l for l in out.splitlines() if "tp4_ep4" in l or l.startswith("case")))

## #4 — small-prefill / mixed backbone(🔬 诊断到底,deferred)

**现象**:≤512 prefill AIC ~50ms vs golden 16ms(~3×);mixed 38.8%

**根因(全部 GPU 验证)**:layerwise **isolated 单 step wall** vs golden **serving busy-bound**
- golden 卡在中间:`busy+comm ~10 < golden 15.5 < isolated wall ~45`(256 tok)
- 缺的不是 kernel,是 **per-step host-dispatch idle**(GPU 等 host 发 kernel):isolated 全暴露(~30ms),serving 多 request overlap 掉 → golden 只剩 **regime-constant host-residual**(captured ~5ms / eager ~30ms,512 处跳,**不随 token 涨**)
- micro-causality:84.6% idle = host-dispatch;layer-count fit `wall=1.61+0.854k`(intercept 才 1.6ms → per-layer 不是 fixed)

**busy-metric VERDICT = 纯 busy 死了**(`runs/BUSY_METRIC_VERDICT.md`)
- Σ(kernel-dur)+comm **欠预测每个 size**(缺 host-residual,非 kernel,不 scale)
- 可行模型 = `busy + comm + regime-host-residual(2 标定常数) + shard-correct MoE`,或 serving full-step,或 calibration

**mixed 22% = 同一家族**:`_get_mix_step_latency` 已是 `context+decode_delta`(max 被显式拒绝)→ 非组装问题;残余在 ctx envelope(512 capture 台阶 + 大 prefill +14%)

In [ ]:
# golden ctx 的 512 capture 台阶 + filter-off 大块（standalone ctx vs AIC）
d = pd.DataFrame({'ctx_tokens':[128,400,496,528,3696],'ctx_kv':[0,3696,528,0,0],
                  'golden_ms':[15.9,16.4,16.2,40.6,43.7],'aic_ms':[37.0,51.5,51.8,39.3,57.4]})
d['err%'] = ((d.aic_ms/d.golden_ms-1)*100).round(0)
print(d.to_string(index=False))
print("\n≤512: golden ~16(captured) AIC 过预测 | >512: golden ~40(eager) AIC 准(-3%/+31%)")
print("golden ctx latency by size:"); print(g[g.phase=='context'].groupby('ctx_tokens').latency_ms.median().round(1).to_string())

## mixed = max(prefill, decode) 的 crossover

- `mixed ≈ max(prefill_term, decode_term)`,但代码已是 `context_total + decode_delta`(更精细,**不是** naive sum,**也不是** max)
- decode_term 封顶 ~8ms;prefill_term ≤512=~16ms(captured 平地板),>512=~40ms+(eager)
- **有 prefill 就 prefill 赢**(16/40 > 8);纯 decode 才 ~5ms

In [ ]:
mix = g[g.phase=='mixed']; dec = g[g.phase=='decode']
print(f"decode 封顶: median {dec.latency_ms.median():.1f}  max {dec.latency_ms.max():.1f} ms")
print("mixed latency vs prefill chunk:")
for b,gg in mix.groupby(pd.cut(mix.ctx_tokens,[0,512,1024,99999]),observed=True):
    print(f"  prefill {str(b):>13}: {gg.latency_ms.median():5.1f} ms")
print("纯 decode: ~5ms  → 512 处跳 16→40ms;有 prefill 就压过 decode")

## backbone vs MoE 误差现状（decode 拆,最可靠）

| 分量 | 误差 | 状态 |
|---|---|---|
| **MoE routed**(expert GEMM) | ~2%(0.157≈0.16) | ✅ 准(#3 修后) |
| MoE router/dispatch | 0(已删) | ✅ 修了 |
| **MoE ep_a2a**(EP all-to-all comm,decode) | **+0.39ms 多加** | ⚠️ single-node 该 overlap≈0,没删干净（可本地摘) |
| MoE-expert GEMM **shard**(EP) | collector 2.93 vs golden(待验) | 🔬 1.82× **可能 mis-framed**（见移交） |
| **backbone**(decode) | +0.45ms(~13% offset) | isolated offset |
| **backbone**(ctx/mixed) | **大头** | isolated wall host-dispatch(主瓶颈) |

→ **MoE 核心(routed)准了;误差现在 = ① backbone isolated-收集(ctx/mixed 大头)② MoE ep_a2a comm 没 overlap(decode 小,可摘)**

## ★ 移交:open items + 下一步

**已闭合**
- #1 skew 自平均(死)· #2 ctx timing(修)· #3 decode 重复计(修+验,51→21%)
- #4 诊断到底:per-layer host-dispatch starvation;纯 busy-metric 死;fix = calibration-flavored

**OPEN / 待验证**
1. **EP shard 1.82×(GPU,prompt 已发,等结果)**:golden 无 EPLB(skew)+ MoE all-to-all 是 barrier → step = **最重 rank**(木桶)。verdict 比的是 collector rank0(power_law 2.93)vs golden rank0(1.61),**可能比错 rank**。查 `serve_nsys_trace` 4 ranks 的 MoE-GEMM busy:**golden 最重 rank ≈ 2.93?** → 是则 collector EP 对、verdict shard 结论要改;否则 collector power_law 过集中
2. **decode ep_a2a comm +0.39ms(本地,快降)**:single-node 该 overlap≈0,没删干净 → 删/overlap → decode 21% 还能降。纯 SDK,不用 GPU
3. **backbone isolated-vs-serving(GPU,calibration)**:fix = busy + 2 regime host-residual 常数(标定 vs golden)+ shard-correct,或 serving full-step。per-model calibration

**下一步候选(按成本)**
- 本地快摘:decode ep_a2a overlap(②)
- 等 GPU:EP rank 验证(①)→ 判 verdict shard 对错
- 大活:backbone calibration(③)— per-model,要 GPU,deferred

**decision**:#3(decode,动吞吐/推荐)已拿。#4/mixed 对 **forward-step-latency API 精度**重要(mixed 38.8% 不小),但要 calibration/GPU + bounded-对推荐 → ROI 待产品定。**"bounded" 只对 config 推荐成立,对 API 精度不成立。**

## 总结

- ✅ **#3 已修**(decode 51→21%,动吞吐,commit + GPU 验)
- 🔬 **#4 诊断到底**:isolated 收集 vs serving busy-bound;纯 busy-metric 死;fix = calibration-flavored,deferred
- **建模规则**:mixed = max(prefill, decode);prefill_term captured 平地板(~16)/ eager(随 size);decode 封顶低(~8)
- **误差现状**:MoE routed 准;剩 ① backbone isolated(ctx/mixed 大头)② MoE ep_a2a comm(decode 小,可摘)③ EP shard(待 rank 验)
- 详见 `SMALL_PREFILL_FIX_DESIGN.md` + `runs/BUSY_METRIC_VERDICT.md`